# Autoresearch Experiment Analysis

Analysis of the richer JEPA-oriented `results.tsv` ledger.

This notebook treats `val_bpb` as the primary metric and uses the extra train/eval columns for diagnosis.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
results_path = Path("results.tsv")
assert results_path.exists(), f"Missing {results_path}"

df = pd.read_csv(results_path, sep="\t")

numeric_columns = [
    "val_bpb",
    "eval_loss",
    "eval_loss_lm",
    "eval_loss_jepa",
    "eval_loss_sigreg",
    "train_loss",
    "train_loss_lm",
    "train_loss_jepa",
    "train_loss_sigreg",
    "lambda_jepa",
    "beta_sigreg",
    "jepa_aux_dropped",
    "jepa_dropout_fraction",
    "peak_memory_gb",
    "tokens_per_sec",
    "tokens_processed",
    "num_steps",
    "wall_seconds",
    "lr",
    "weight_decay",
    "batch_size",
    "seq_len",
    "train_shards",
    "eval_batches",
]

for column in numeric_columns:
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

df["status"] = df["status"].fillna("pending").astype(str).str.strip().str.upper()
df["description"] = df["description"].fillna("").astype(str)
df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], errors="coerce")

kept_like_status = {"BASELINE", "KEEP"}
print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
counts = df["status"].value_counts(dropna=False)
print("Experiment outcomes:")
print(counts.to_string())

n_keep = counts.get("KEEP", 0)
n_baseline = counts.get("BASELINE", 0)
n_discard = counts.get("DISCARD", 0)
n_crash = counts.get("CRASH", 0)
n_decided = n_baseline + n_keep + n_discard
if n_decided > 0:
    print(f"\nAdvance rate: {(n_baseline + n_keep)}/{n_decided} = {(n_baseline + n_keep) / n_decided:.1%}")

In [ ]:
kept = df[df["status"].isin(kept_like_status)].copy()
print(f"Frontier experiments ({len(kept)} total):\n")
for i, row in kept.iterrows():
    print(
        f"  #{i:3d}  status={row['status']:<8s}  bpb={row['val_bpb']:.6f}  "
        f"mem={row['peak_memory_gb']:.2f}GB  lm={row['eval_loss_lm']:.4f}  {row['description']}"
    )

## Val BPB Over Time

The running minimum over `BASELINE` and `KEEP` rows is the active frontier.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

valid = df[df["status"] != "CRASH"].copy().reset_index(drop=True)
assert len(valid) > 0, "No non-crash rows to plot"
baseline_bpb = valid.loc[0, "val_bpb"]

disc = valid[valid["status"] == "DISCARD"]
ax.scatter(
    disc.index,
    disc["val_bpb"],
    c="#cccccc",
    s=12,
    alpha=0.5,
    zorder=2,
    label="Discarded",
)

frontier_rows = valid[valid["status"].isin(kept_like_status)]
frontier_idx = frontier_rows.index
frontier_bpb = frontier_rows["val_bpb"]
frontier_running_min = frontier_bpb.cummin()

baseline_rows = frontier_rows[frontier_rows["status"] == "BASELINE"]
keep_rows = frontier_rows[frontier_rows["status"] == "KEEP"]

ax.scatter(
    baseline_rows.index,
    baseline_rows["val_bpb"],
    c="#3498db",
    s=60,
    zorder=4,
    label="Baseline",
    edgecolors="black",
    linewidths=0.5,
)
ax.scatter(
    keep_rows.index,
    keep_rows["val_bpb"],
    c="#2ecc71",
    s=50,
    zorder=4,
    label="Kept",
    edgecolors="black",
    linewidths=0.5,
)
ax.step(
    frontier_idx,
    frontier_running_min,
    where="post",
    color="#27ae60",
    linewidth=2,
    alpha=0.8,
    zorder=3,
    label="Running best",
)

for idx, row in frontier_rows.iterrows():
    desc = row["description"].strip() or row["run_name"]
    if len(desc) > 48:
        desc = desc[:45] + "..."
    ax.annotate(
        desc,
        (idx, row["val_bpb"]),
        textcoords="offset points",
        xytext=(6, 6),
        fontsize=8,
        color="#1a7a3a",
        alpha=0.9,
        rotation=28,
        ha="left",
        va="bottom",
    )

best_bpb = frontier_bpb.min()
margin = max((baseline_bpb - best_bpb) * 0.15, 1e-4)
ax.set_ylim(best_bpb - margin, baseline_bpb + margin)
ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel("Validation BPB (lower is better)", fontsize=12)
ax.set_title(
    f"Autoresearch Progress: {len(df)} Experiments, {len(frontier_rows)} Frontier States",
    fontsize=14,
)
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to progress.png")

## Summary Statistics

In [ ]:
frontier = df[df["status"].isin(kept_like_status)].copy()
assert len(frontier) > 0, "Need at least one BASELINE or KEEP row"

baseline_row = frontier.iloc[0]
best_row = frontier.loc[frontier["val_bpb"].idxmin()]
baseline_bpb = baseline_row["val_bpb"]
best_bpb = best_row["val_bpb"]

print(f"Baseline val_bpb:  {baseline_bpb:.6f}")
print(f"Best val_bpb:      {best_bpb:.6f}")
print(f"Total improvement: {baseline_bpb - best_bpb:.6f} ({(baseline_bpb - best_bpb) / baseline_bpb * 100:.2f}%)")
print(f"Best experiment:   {best_row['description']}")
print(f"Best eval LM:      {best_row['eval_loss_lm']:.6f}")
print(f"Best memory GB:    {best_row['peak_memory_gb']:.3f}")
print()
print("Frontier progression:")
for idx, row in frontier.reset_index().iterrows():
    print(
        f"  Experiment #{row['index']:3d}: status={row['status']:<8s} "
        f"bpb={row['val_bpb']:.6f} lm={row['eval_loss_lm']:.4f} {row['description']}"
    )

## Top Frontier Improvements

In [ ]:
frontier = df[df["status"].isin(kept_like_status)].copy()
frontier["prev_bpb"] = frontier["val_bpb"].shift(1)
frontier["delta"] = frontier["prev_bpb"] - frontier["val_bpb"]
hits = frontier.iloc[1:].copy().sort_values("delta", ascending=False)

print(f"{'Rank':>4}  {'Delta':>8}  {'BPB':>10}  {'LM':>10}  Description")
print("-" * 100)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    print(
        f"{rank:4d}  {row['delta']:+.6f}  {row['val_bpb']:.6f}  "
        f"{row['eval_loss_lm']:.6f}  {row['description']}"
    )

if len(hits) > 0:
    print(f"\n{'':>4}  {hits['delta'].sum():+.6f}  {'':>10}  {'':>10}  TOTAL improvement over baseline")